In [1]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Correlation across all genes

In [2]:
# Configuration and paths
mac = 20
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

abs_phenotypes = True

# Load annotation configuration
if abs_phenotypes:
    config_path = "/home/dnanexus/ukbgym/config_wgs.yaml" # If abs_pheno then deleteriousness direction
else:    
    config_path = "/home/dnanexus/ukbgym/config_olink.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i8
"""plof""","""loftee_hc""","""#E31A1C""","""LOFTEE HC""",1
"""plof_consequences""","""consequence_frameshift_variant""","""#E31A1C""","""VEP Frameshift""",1
"""plof_consequences""","""consequence_stop_gained""","""#FB9A99""","""VEP Stop Gained""",1
"""plof_consequences""","""consequence_splice_donor_varia…","""#6A1B9A""","""VEP Splice Donor""",1
"""plof_consequences""","""consequence_splice_acceptor_va…","""#AB47BC""","""VEP Splice Acceptor""",1
…,…,…,…,…
"""vep_consequences""","""consequence_splice_acceptor_va…","""#AB47BC""","""VEP Splice Acceptor""",1
"""vep_consequences""","""consequence_start_lost""","""#E65100""","""VEP Start Lost""",1
"""vep_consequences""","""consequence_stop_lost""","""#FFB300""","""VEP Stop Lost""",1


In [3]:
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/annotation_files/genebass352genes_olink371genes_annotated_251205.parquet -o /home/dnanexus/data_dir/genebass352genes_olink371genes_annotated_251205.parquet

# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/annotation_files/genebass352genes_olink371genes_annotated_251205_fillna.parquet -o /home/dnanexus/data_dir/genebass352genes_olink371genes_annotated_251205_fillna.parquet

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym.parquet -o /home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet

Error: path "/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet"
already exists but -f/--overwrite was not set


In [4]:
anno = pl.scan_parquet("/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet")

min_range = -2000
max_range = +0

anno = (
    anno
    .filter(
        # Choose CDS
        (pl.col('vep_cds_relaxed')==True) &
        # ((pl.col('vep_cds_relaxed')==True) | (pl.col('mane_cds')==True)) &
        # (pl.col('non_mane_cds')==False) &

        # Choose non CDS only
        # ((pl.col('vep_cds_relaxed')==False) & (pl.col('mane_cds')==False) & (pl.col('non_mane_cds')==False)) &

        # Choose Gene Body
        # ((pl.col('consequence_upstream_gene_variant') == False) & (pl.col('consequence_downstream_gene_variant') == False)) &

        # Choose VEP consequence
        # (pl.col('consequence_missense_variant') == True) &
        # (pl.col('consequence_synonymous_variant') == True) &
        # (pl.col('consequence_5_prime_utr_variant') == True) &
        # (pl.col('consequence_upstream_gene_variant') == True) &
        # (pl.col('consequence_downstream_gene_variant') == True) &
        # (pl.col('consequence_intron_variant') == True) &

        # Choose MobiDB region
        # (pl.col('mobi_lip_full') == True) &
        # (pl.col('mobi_disorder_full') == True) &

        # Custom variant class filter
        # filter_expression &

        # Choose regulatory region
        # (pl.col('encode_eh_pr') == True) &
        # (pl.col('encode_all_tf') == True) &
        
        # Proximity to TSS
        # (pl.col('dist_to_tss') >= min_range) &
        # (pl.col('dist_to_tss') <= max_range) &

        # Only SNPs
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1)
    )
    # .with_columns(
    #     core_promoter = pl.col('dist_to_tss').abs() <= 50,
    #     encode_annotated = pl.col('not_in_encode') == False,
    #     encode_eh_pr = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pls', 'encode_pels', 'encode_dels']),
    #     encode_all_tf = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_tf', 'encode_ca_tf']),
    # )
)

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

anno = (
    anno
    .select(
        # set(['id', 'region', 'tss', 'strand', 'gene_length', 'gene_name', 'dist_to_tss']).union(set(existing_annos))
        set(['id', 'region']).union(set(existing_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_ustop_gained,absplice_dna_max,consequence_splice_acceptor_variant,ted_domain,consequence_stop_gained,promoterai,score_pai3d,mobi_curated_disorder_priority,cadd_raw,delta_score,abexp_abs_max,loftee_lc,mobi_lip_full,region,consequence_stop_lost,five_prime_utr_variant_consequence_ustop_lost,five_prime_utr_variant_consequence_uaug_gained,loftee_hc,absplice2_max,low_complexity_domain,consequence_missense_variant,consequence_synonymous_variant,consequence_frameshift_variant,verphylop,consequence_splice_donor_variant,pangolin_score,consequence_start_lost,esmscoremissense,am_pathogenicity,gpn_score,id
u8,u8,f32,i8,bool,i8,f32,f32,bool,f32,f32,f32,i8,bool,str,i8,u8,u8,i8,f32,bool,i8,i8,i8,f32,i8,f32,i8,f32,f32,f32,str
0,0,0.003,0,false,0,0.0,0.699444,false,3.681756,0.05,0.060269,0,false,"""ENSG00000145715""",0,0,0,0,0.000327,false,1,0,0,8.687,0,0.03,0,-4.246,0.2063,-11.54,"""chr5:87376465:A:T"""
0,0,0.003,0,false,1,0.0,0.0,false,7.69086,0.0,0.946601,0,true,"""ENSG00000120341""",0,0,0,1,0.002165,false,0,0,0,2.372,0,0.06,0,0.0,0.0,-1.6,"""chr1:177933615:G:A"""
0,0,0.001,0,false,0,-0.143,0.0,false,0.528766,0.0,0.012463,0,false,"""ENSG00000168763""",0,0,0,0,0.000033,false,0,1,0,-1.5,0,0.0,0,0.0,0.0,0.36,"""chr2:96816550:C:T"""
0,0,0.0,0,false,0,0.0,0.0,false,3.032575,0.08,0.015385,0,true,"""ENSG00000155657""",0,0,0,0,0.000034,false,1,0,0,6.948,0,0.03,0,-6.253,0.0,-8.09,"""chr2:178598858:G:A"""
0,0,0.003,0,true,0,0.0,0.550982,false,4.428444,0.0,0.160049,0,false,"""ENSG00000136720""",0,0,0,0,0.000321,false,1,0,0,-0.102,0,0.0,0,-5.721,0.2002,-5.12,"""chr2:128318064:C:T"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,0,0.003,0,true,1,0.0032,0.0,false,5.248261,0.01,1.39091,0,false,"""ENSG00000004468""",0,0,0,1,0.002172,false,0,0,0,-0.6,0,0.03,0,0.0,0.0,-6.74,"""chr4:15816634:C:A"""
0,0,0.001,0,false,0,-0.0234,0.0,false,1.2186,0.0,0.013111,0,false,"""ENSG00000167202""",0,0,0,0,0.000033,true,0,1,0,0.709,0,0.0,0,0.0,0.0,-4.14,"""chr15:78077599:C:A"""
0,0,0.001,0,true,0,0.0,0.0,false,0.213689,0.0,0.006189,0,true,"""ENSG00000072121""",0,0,0,0,0.000033,false,0,1,0,2.934,0,0.0,0,0.0,0.0,-0.88,"""chr14:67783209:G:A"""


In [5]:
selected_annos = anno_config_df.filter(
    (pl.col('annotation') == 'loftee_hc') # LOFTEE
)['annotation'].to_list()

melted_anno = (
    anno
    
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    ).with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )
)
melted_anno

id,region,annotation,annotation_score
str,str,str,f32
"""chr5:87376465:A:T""","""ENSG00000145715""","""loftee_hc""",0.0
"""chr1:177933615:G:A""","""ENSG00000120341""","""loftee_hc""",1.0
"""chr2:96816550:C:T""","""ENSG00000168763""","""loftee_hc""",0.0
"""chr2:178598858:G:A""","""ENSG00000155657""","""loftee_hc""",0.0
"""chr2:128318064:C:T""","""ENSG00000136720""","""loftee_hc""",0.0
…,…,…,…
"""chr4:15816634:C:A""","""ENSG00000004468""","""loftee_hc""",1.0
"""chr15:78077599:C:A""","""ENSG00000167202""","""loftee_hc""",0.0
"""chr14:67783209:G:A""","""ENSG00000072121""","""loftee_hc""",0.0


In [6]:
# Subset Olink RVAT significant

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/blacklist/proteomics_prs_am_loftee_mac20_burden_regression_results.parquet -o /home/dnanexus/data_dir/olink/proteomics_prs_am_loftee_mac20_burden_regression_results.parquet

olink_whitelist = (
    pl.read_parquet('/home/dnanexus/data_dir/olink/proteomics_prs_am_loftee_mac20_burden_regression_results.parquet')
    .rename({'gene': 'region'})
    .filter(pl.col('padj_perm')<=0.05)
    .select(['region'])
    .with_columns(
        phenotype = pl.col('region') + '_olink'
    )
)

olink_whitelist

Error: path "/home/dnanexus/data_dir/olink/proteomics_prs_am_loftee_mac20_burd
en_regression_results.parquet" already exists but -f/--overwrite was not set


region,phenotype
str,str
"""ENSG00000213066""","""ENSG00000213066_olink"""
"""ENSG00000107201""","""ENSG00000107201_olink"""
"""ENSG00000113739""","""ENSG00000113739_olink"""
"""ENSG00000085514""","""ENSG00000085514_olink"""
"""ENSG00000071051""","""ENSG00000071051_olink"""
…,…
"""ENSG00000092529""","""ENSG00000092529_olink"""
"""ENSG00000145730""","""ENSG00000145730_olink"""
"""ENSG00000135218""","""ENSG00000135218_olink"""


In [7]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/olink_all_genes_EURunrelated_appv_percentiles.parquet -o /home/dnanexus/olink_all_genes_EURunrelated_appv_percentiles.parquet

# Read Olink phenotype data
olink_appv = pl.scan_parquet("/home/dnanexus/olink_all_genes_EURunrelated_appv_percentiles.parquet")

# Merge phenotype data and annotation data
gp_corr_df = (
    olink_appv
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    # .join(
    #     olink_whitelist.lazy(),
    #     on=["region", "phenotype"],
    #     how="semi"
    # )
    .with_columns(
        mean_pheno_value = pl.when(abs_phenotypes)
        .then(pl.col('mean_pheno_value').abs())
        .otherwise(pl.col('mean_pheno_value'))
    )
    .join(
        melted_anno.lazy(), 
        on="id", 
        how="inner"
    )
    .filter(
        pl.col('region') + '_olink' == pl.col('phenotype')
    )
    .with_columns(
        pl.col(c)
        .rank("average")
        .over(["region", "phenotype", "annotation"])
        .alias(f"{c}_rank")
        for c in ['mean_pheno_value', 'annotation_score']
    )
    .group_by(["region", "phenotype", "annotation"])
    .agg(
        n_variants = pl.col("id").count(),
        correlation = pl.corr("annotation_score_rank", "mean_pheno_value_rank", propagate_nans=True)
    )
    .collect(engine='streaming')
)

gp_corr_df

Error: path
"/home/dnanexus/olink_all_genes_EURunrelated_appv_percentiles.parquet" already
exists but -f/--overwrite was not set


region,phenotype,annotation,n_variants,correlation
str,str,str,u64,f64
"""ENSG00000163606""","""ENSG00000163606_olink""","""loftee_hc""",67,0.204082
"""ENSG00000144857""","""ENSG00000144857_olink""","""loftee_hc""",337,0.162616
"""ENSG00000136698""","""ENSG00000136698_olink""","""loftee_hc""",6,NaN
"""ENSG00000133661""","""ENSG00000133661_olink""","""loftee_hc""",101,NaN
"""ENSG00000051180""","""ENSG00000051180_olink""","""loftee_hc""",56,NaN
…,…,…,…,…
"""ENSG00000158793""","""ENSG00000158793_olink""","""loftee_hc""",190,0.318832
"""ENSG00000137101""","""ENSG00000137101_olink""","""loftee_hc""",69,0.176579
"""ENSG00000111241""","""ENSG00000111241_olink""","""loftee_hc""",61,NaN


In [9]:
gp_corr_df = (
    gp_corr_df
    .select(['region', 'phenotype', 'annotation', 'n_variants', 'correlation'])
    .join(
        anno_config_df,
        on='annotation'
    )
    .with_columns(
        corr_beta = pl.col('correlation')*pl.col('annotation_dir')
    )
)

gp_corr_df

region,phenotype,annotation,n_variants,correlation,category,color,label,annotation_dir,corr_beta
str,str,str,u64,f64,str,str,str,i8,f64
"""ENSG00000163606""","""ENSG00000163606_olink""","""loftee_hc""",67,0.204082,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.204082
"""ENSG00000144857""","""ENSG00000144857_olink""","""loftee_hc""",337,0.162616,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.162616
"""ENSG00000136698""","""ENSG00000136698_olink""","""loftee_hc""",6,NaN,"""plof""","""#E31A1C""","""LOFTEE HC""",1,NaN
"""ENSG00000133661""","""ENSG00000133661_olink""","""loftee_hc""",101,NaN,"""plof""","""#E31A1C""","""LOFTEE HC""",1,NaN
"""ENSG00000051180""","""ENSG00000051180_olink""","""loftee_hc""",56,NaN,"""plof""","""#E31A1C""","""LOFTEE HC""",1,NaN
…,…,…,…,…,…,…,…,…,…
"""ENSG00000158793""","""ENSG00000158793_olink""","""loftee_hc""",190,0.318832,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.318832
"""ENSG00000137101""","""ENSG00000137101_olink""","""loftee_hc""",69,0.176579,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.176579
"""ENSG00000111241""","""ENSG00000111241_olink""","""loftee_hc""",61,NaN,"""plof""","""#E31A1C""","""LOFTEE HC""",1,NaN


In [10]:
gp_corr_df.filter(pl.col('annotation')=='loftee_hc').write_parquet('/home/dnanexus/data_dir/olink_all_mac20_lofteeHC_correlations.parquet')

In [11]:
!dx upload /home/dnanexus/data_dir/olink_all_mac20_lofteeHC_correlations.parquet --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/olink_all_mac20_lofteeHC_correlations.parquet

[===========================================================>] Uploaded 61,016 of 61,016 bytes (100%) /home/dnanexus/data_dir/olink_all_mac20_lofteeHC_correlations.parquet
ID                                file-J5jVVpjJg0y70J1P4Vkv6q6z
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/REGENIE_results
Name                              olink_all_mac20_lofteeHC_correlations.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Wed Jan 21 18:20:27 2026
Created by                        shubhankar
 via the job                      job-J5jKzb8Jg0yGx8PvvjFky1Xy
Last modified                     Wed Jan 21 18:20:28 2026
Media type                        
archivalState          